<a href="https://colab.research.google.com/github/cbonnin88/SaaS_Pulse/blob/main/saas_pulse.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
!pip install streamlit
!pip install pyngrok

In [23]:
import polars as pl
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# **Creating the Dataset**

In [24]:
def generate_saas_data(n_customers=4000):
  np.random.seed(42)
  start_date = datetime(2024,1,1)

  # 1. Customers
  segments = ['Startup','SMB','Enterprise']
  cust_data = {
      'customer_id': [f'user_{i:04d}' for i in range(n_customers)],
      'signup_date': [start_date + timedelta(days=np.random.randint(0,450)) for _ in range(n_customers)],
      'segment': np.random.choice(segments, n_customers, p=[0.5,0.3,0.2]),
      'cac': np.random.choice([200,600,3000], n_customers),
      'cogs_pct': np.random.uniform(0.15,0.25,n_customers)
  }

  # 2. Transactions
  transactions = []
  churn_labels = []

  for i in range(n_customers):
    segment = cust_data['segment'][i]
    signup_date = cust_data['signup_date'][i]
    customer_id = cust_data['customer_id'][i]
    cogs_pct = cust_data['cogs_pct'][i]

    churn_rate = 0.04 if segment == 'Enterprise' else 0.15
    lifespan = min(np.random.geometric(p=churn_rate),24)
    churn_labels.append({'customer_id':customer_id,'churned': 1 if lifespan < 24 else 0})

    base_price = 500 if segment == 'Enterprise' else 99

    for month in range(lifespan):
      tx_date = signup_date + pd.DateOffset(months=month) if 'pd' in globals() else signup_date + timedelta(days=30.44 * month)
      if tx_date > datetime(2026,5,1): break

      if np.random.random() < 0.10: base_price *= 1.2

      transactions.append({
          'customer_id': customer_id,
          'date': tx_date,
          'amount': base_price,
          'margin_amount': base_price * (1 - cogs_pct)
      })

  # Saving using Polars
  pl.DataFrame(cust_data).write_csv('customer_csv')
  pl.DataFrame(transactions).write_csv('subscriptions.csv')
  pl.DataFrame(churn_labels).write_csv('churn_data.csv')
  print('SaaS Dataset Generated')

In [25]:
generate_saas_data()

SaaS Dataset Generated


# **Streamlit App Section**

In [48]:
%%writefile app.py
import streamlit as st
import polars as pl
import pandas as pd
import plotly.express as px
from sklearn.ensemble import RandomForestClassifier

st.set_page_config(page_title='Executive Analytics',layout='wide')

@st.cache_data
def load_data():
  customers = pl.read_csv('customer_csv',try_parse_dates=True)
  subscriptions = pl.read_csv('subscriptions.csv',try_parse_dates=True)
  churn = pl.read_csv('churn_data.csv')

  # Joining Dataframes
  df_product = subscriptions.join(customers,on='customer_id')

  # Data manipulation
  df_product = df_product.with_columns([
      pl.col('signup_date').dt.truncate('1mo').alias('signup_month'),
      pl.col('date').dt.truncate('1mo').alias('tx_month')
  ])

  df_product = df_product.with_columns(
      ((pl.col('tx_month').dt.year() - pl.col('signup_month').dt.year())* 12 +
       (pl.col('tx_month').dt.month() * pl.col('signup_month').dt.month())).alias('cohort_index')
  )
  return df_product, churn, customers

df_product, churn_df, raw_cust = load_data()

# --- ML CHURN MODEL ---
@st.cache_resource
def get_predictions(data, labels):
  feat = data.group_by('customer_id').agg([
      pl.col('amount').mean().alias('avg_spend'),
      pl.col('amount').sum().alias('total_rev'),
      pl.col('cohort_index').max().alias('tenure')
  ])

  # Sklearn requires Pandas/Numpy
  train_pd = feat.join(labels, on='customer_id').to_pandas()

  model = RandomForestClassifier(n_estimators=100, random_state=42)
  X = train_pd[['avg_spend','total_rev','tenure']]
  model.fit(X,train_pd['churned'])

  train_pd['churn_probability'] = model.predict_proba(X)[:, 1]

  return pl.DataFrame(train_pd)

prediction_results = get_predictions(df_product, churn_df)

# --- UI ---
st.title('SaaS Analytics')
risk_threshold_pct = st.sidebar.slider(
    'Churn Sensitivity',
    min_value=0,
    max_value=100,
    value=75,
    step=5,
    format='%d%%', # Adding the % sign
    help='Target users with a probability of churning hirhg than this'
)
risk_threshold = risk_threshold_pct / 100

tab_nrr, tab_payback, tab_rader = st.tabs(['Net Revenue Retention (NRR)','Margin Adjusted Payback','Churn Risk Radar'])

with tab_nrr:
  st.subheader('Cohort Net Revenue Retention')
  rev_counts = df_product.group_by('signup_month','cohort_index').agg(pl.col('amount').sum())

  # Isolate Month 0 to calculate NRR denominator
  base_rev = (
      rev_counts
      .sort(['signup_month','cohort_index'])
      .group_by('signup_month')
      .first()
      .select(['signup_month',pl.col('amount').alias('base')])
  )

  nrr_df = rev_counts.join(base_rev, on='signup_month').with_columns(
      (pl.col('amount') / pl.col('base')).alias('nrr')
  )

  # Pivoting and converting to pandas for heatmap formatting
  nrr_pivot = nrr_df.pivot(index='signup_month',on='cohort_index',values='nrr').sort('signup_month')
  nrr_pd = nrr_pivot.to_pandas().set_index('signup_month')

  nrr_pd = nrr_pd.reindex(sorted(nrr_pd.columns), axis=1)

  fig_nrr = px.imshow(
      nrr_pd,
      text_auto='.0%',
      color_continuous_scale='viridis_r',
      y=nrr_pd.index.strftime('%Y-%m'),
      color_continuous_midpoint=1.0,
      labels=dict(x='Months Since Signup', y='Cohort Month',color='NRR')
  )

  st.plotly_chart(fig_nrr, use_container_width=True)
  st.caption("Values > 100% indicate expansion revenue is outpacing churn.")

with tab_payback:
  st.subheader('Payback Analysis (Revenue vs. Gross Margin)')

  total_customers = raw_cust.select(pl.col('customer_id').n_unique()).item()
  avg_cac = raw_cust.select(pl.col('cac').mean()).item()

  # Cumulative Sums
  clv_curve = df_product.group_by('cohort_index').agg([
      pl.col('amount').sum().alias('rev'),
      pl.col('margin_amount').sum().alias('margin')
  ]).sort('cohort_index')

  clv_curve = clv_curve.with_columns([
      (pl.col('rev') / total_customers).cum_sum().alias('cum_rev'),
      (pl.col('margin') / total_customers).cum_sum().alias('cum_margin')
  ]).to_pandas()

  fig_pb = px.line(
      labels={'value':'EUR (€)','cohort_index':'Months'}
  )
  fig_pb.add_scatter(x=clv_curve['cohort_index'],y=clv_curve['cum_rev'],name='Cumulative Revenue')
  fig_pb.add_scatter(x=clv_curve['cohort_index'],y=clv_curve['cum_margin'],name='Cumulative Margin')
  fig_pb.add_hline(y=avg_cac, line_dash='dash',line_color='red',annotation_text='Avg CAC')
  st.plotly_chart(fig_pb, use_container_width=True)

  with tab_rader:
    st.subheader('Predictive Outreach List')
    at_risk = prediction_results.filter(pl.col('churn_probability') >= risk_threshold).sort('churn_probability',descending=True)

    col1, col2 = st.columns([1,4])
    col1.metric('Flagged Users', at_risk.height)

    csv = at_risk.write_csv()
    st.download_button('📥 Export CSV for CS Team', data=csv, file_name='at_risk.csv',mime='text/csv')

    # Streamlit natively renders Polars dataframes
    st.dataframe(at_risk, use_container_width=True)

Overwriting app.py


In [49]:
from pyngrok import ngrok
import os
import time
from google.colab import userdata

os.system("pkill -f streamlit")
ngrok.kill()

NGROK_TOKEN = userdata.get('ngrok_token')
os.system(f"ngrok config add-authtoken {NGROK_TOKEN}")

# 3. Start Streamlit and FORCE it to use port 8501
print("Starting Streamlit...")
os.system("streamlit run app.py --server.port 8501 &")

# 4. Wait for 5 seconds to give Streamlit time to fully boot up
time.sleep(5)

# 5. Create a fresh tunnel
try:
    public_url = ngrok.connect(8501)
    print(f"✅ SaaS Pulse Dashboard is Live! Click here: {public_url.public_url}")
except Exception as e:
    print(f"Failed to create tunnel: {e}")

Starting Streamlit...
✅ SaaS Pulse Dashboard is Live! Click here: https://5454-34-58-41-126.ngrok-free.app
